In [1]:
import uproot
import matplotlib.pyplot as plt
import pandas as pd
import tqdm
import numpy as np
import os
import glob
import awkward as ak
import boost_histogram as bh
import mplhep
from matplotlib.colors import LogNorm
# !pip install pyarrow
# !pip install "pyarrow<21" --break-system-packages

In [2]:
runs_F222 = [7733, 8712]
runs_F223 = [8713, 9188]

runs_F231 = [10555, 11470]
runs_F232 = [11470, 11703]

runs_F241 = [14399, 15026]
runs_tungsten = [15030, 15594]
runs_F242 = [15652, 15820]
runs_caloNu = [15821, 16924]
runs_F243 = [16928, 17230]

runs_2022 = [8025, 9188]
runs_2023 = [10604, 11703]
runs_2024_preCaloNu = [14399, 15821]
runs_2024_CaloNu = [15821, 16924]
runs_2024_afterCaloNu = [16928, 17230]

In [3]:
files = ["build/200093.root:nt",
         "build/200094.root:nt",
         "build/200100.root:nt",]

In [4]:
data = uproot.concatenate(files, library="pd")

KeyboardInterrupt: 

In [5]:
def get_cutflow_and_yields_for_run_range(files, run_start=None, run_end=None, mass=1.1):
    cutflow_dict = {}
    run_yields = {}

    total_lumi = 0
    for fpath in tqdm.tqdm(files):

        run = int(os.path.basename(fpath).split("/")[-1].split(".")[0])
        # if run not in good_runs: continue
        
        
        if run_start is not None and run_start > run:
            continue
        if run_end is not None and run > run_end:
            continue

        with uproot.open(fpath) as f:
            tree = f["cutflow"]
            lumi = f["meta"].arrays(library="pd")["lumi"][0]
            df = tree.arrays(library="pd")

            run = int(os.path.basename(fpath).split("/")[-1].split(".")[0])
            # if run not in good_runs: continue

            
            total_lumi += lumi[0] / 1000 # /pb -> /fb
            for idx, row in df.iterrows():
                cut = row["cut_name"]
                all = row["all"]
                passed = row["passed"]

                if cut not in cutflow_dict:
                    cutflow_dict[cut] = {"all": 0, "passed": 0}

                cutflow_dict[cut]["all"] += all
                cutflow_dict[cut]["passed"] += passed
                cutflow_dict[cut]["efficiency"] = cutflow_dict[cut]["passed"] / cutflow_dict[cut]["all"] if cutflow_dict[cut]["all"] > 0 else 0
                cutflow_dict[cut]["cumulative_efficiency"] = cutflow_dict[cut]["passed"] / cutflow_dict[list(cutflow_dict.keys())[0]]["all"] if cutflow_dict[list(cutflow_dict.keys())[0]]["all"] > 0 else 0
                cutflow_dict[cut]["efficiency_error"] = np.sqrt(cutflow_dict[cut]["efficiency"] * (1 - cutflow_dict[cut]["efficiency"]) / cutflow_dict[cut]["all"]) if cutflow_dict[cut]["all"] > 0 else 0


                final_yield = np.amin(df["passed"])
                run_yields[int(run)] = final_yield

    if total_lumi <= 0:
        total_lumi = 10_000

    print(f"Total lumi for runs {run_start} to {run_end}: {total_lumi:.2f} fb^-1")

    cutflow = pd.DataFrame.from_dict(cutflow_dict, orient="index")
    cutflow["rate [events/fb/tonne]"] = cutflow["passed"] / (total_lumi * mass) if total_lumi > 0 else 0
    cutflow = cutflow.sort_values(by='passed', ascending=False)
            
    return cutflow, run_yields

In [6]:
cutflow_df, run_yields = get_cutflow_and_yields_for_run_range(files = ["build/200093.root", "build/200094.root", "build/200100.root",], mass=1.1)

100%|██████████| 3/3 [00:00<00:00, 52.44it/s]

Total lumi for runs None to None: 10000.00 fb^-1


In [7]:
# cutflow for MC 2024 with caloNu:

# Cut                            Events Before   Events After    Efficiency (%)  Cumalative Efficiency (%)
# Remove CaloNu PMT region (truth) 1234895         1232337         99.79           99.79               
# isCC                           1232337         933845          75.78           75.62               
# is Nu mu                       933845          778117          83.32           63.01               
# in fiducial volume             778117          197115          25.33           15.96               
# has truth dec r < 100 cm       197115          95200           48.30           7.71                
# has truth pz > 100 GeV         95200           87910           92.34           7.12                
# reduced_VetoNu0_charge         87910           78442           89.23           6.35                
# veto charge cut > 40 pC        78442           78345           99.88           6.34                
# Preshower > 2.5 pC             78345           67850           86.60           5.49                
# Track pz > 100 GeV             67850           35551           52.40           2.88                
# Track layers >= 7              35551           34822           97.95           2.82                
# Track nDoF >= 9                34822           32997           94.76           2.67                
# Track chi2/ndf < 15            32997           32987           99.97           2.67                
# Track r at max radius < 95 cm  32987           23473           71.16           1.90                
# Track rIFT < 95 cm             23473           23044           98.17           1.87                
# Track rVetoNu < 120 cm         23044           22768           98.80           1.84                
# Track theta < 25 mrad          22768           21982           96.55           1.78                
# One long track                 21982           21982           100.00          1.78                
# Timing charge > 20 pC          21982           21979           99.99           1.78                
# reduced charge < 30 pC         21979           21979           100.00          1.78                

In [8]:
# cutflow_df = pd.DataFrame(cutflow_dict).T
# cutflow_df["efficiency"] = (cutflow_df["passed"] / cutflow_df["all"] * 100).round(2)
# cutflow_df = cutflow_df.reset_index().rename(columns={"index": "cut_name"})
display(cutflow_df)

,all,passed,efficiency,cumulative_efficiency,efficiency_error,rate [events/fb/tonne]
Remove CaloNu PMT region (truth),1450841,1450841,1.000000,1.000000,0.000000,131.894636
CC events only,1450841,1100511,0.758533,0.758533,0.000355,100.046455
nu_mu only,1100511,918140,0.834285,0.632833,0.000354,83.467273
In Faser Nu Box or Lead Block,918140,358654,0.390631,0.247204,0.000509,32.604909
Truth dec r < 100 mm,358654,173518,0.483803,0.119598,0.000834,15.774364
Truth pz > 100 GeV,173518,160243,0.923495,0.110448,0.000638,14.567545
VetoNu0 reduced charge < 30 pC,160243,145902,0.910505,0.100564,0.000713,13.263818
VetoNu1 reduced charge < 30 pC,145902,139183,0.953949,0.095933,0.000549,12.653000
Veto20 and Veto21 charge > 40 pC,139183,138142,0.992521,0.095215,0.000231,12.558364
Timing Station Charge > 20 pC,138142,134099,0.970733,0.092428,0.000453,12.190818


In [9]:
def wrap_math_text(s, caption="My table"):
    s = s.replace(r"\begin{table}", "")
    s = s.replace(r"\end{table}", "")
    s = s.replace(r"<", r"$<$").replace(r">", r"$>$").replace("=", r"$=$").replace("+", r"$+$").replace("-", r"$-$").replace("*", r"$*$").replace("/", r"$/$").replace("<=", r"$\leq$").replace(">=", r"$\geq$").replace("!=", r"$\neq$")
    s = rf"""
\begin{{table}}
\centering
\caption{{{caption}}}
\begin{{adjustbox}}{{max width=\textwidth}}
{s}
\end{{adjustbox}}
\end{{table}}
"""
    return s

In [10]:

def write_cutflow_to_latex(cutflow_df, filename="cutflow.tex", caption="Cutflow"):
    
    with open(filename, 'w') as f:

        longtable = False
        float_format = "{:0.2f}".format
        colums = ["passed", "efficiency", "cumulative_efficiency", "rate [events/fb/tonne]"]

        f.write(r"\documentclass{article}" + "\n")
        f.write(r"\usepackage{graphicx}" + "\n")
        f.write(r"\usepackage{booktabs}" + "\n")
        f.write(r"\usepackage{longtable}" + "\n")
        f.write(r"\usepackage{adjustbox}" + "\n")
        f.write(r"\begin{document}" + "\n")
        t = wrap_math_text(cutflow_df.to_latex(escape=True, columns=colums, longtable=longtable, float_format=float_format), caption=caption)
        f.write(t + "\n")
        f.write(r"\end{document}" + "\n") 


In [11]:
write_cutflow_to_latex(cutflow_df, filename="cutflow_NocaloNuMC.tex", caption="Cutflow for MC 2024 with NocaloNu")

In [12]:
files = ["build/200139.1.root",
         "build/200140.1.root",
         "build/200146.1.root",]

hists = {}
for file in files:
    with uproot.open(file) as f:
        hists_dir = f["histograms"]
        for hist_name in hists_dir.keys():
            hist = hists_dir[hist_name].to_boost()
            if hist_name not in hists:
                hists[hist_name] = hist
            else:
                hists[hist_name] += hist

FileNotFoundError: [Errno 2] No such file or directory: '/opt/ppd/atlas/bewilson/nuAnalysis/build/200139.1.root'

In [ ]:
for hist_name, hist in hists.items():
    fig, ax = plt.subplots(figsize=(10, 6))
    try:
        mplhep.histplot(hist, ax=ax)
    except Exception as e:
        mplhep.hist2dplot(hist, ax=ax, norm=LogNorm())
    ax.set_title(hist_name)

In [ ]:
nu_vz = np.array([x[0] for x in data["truth_dec_z"].to_numpy()])

In [ ]:
caloNu_tot_EM = data["CaloNu_total_E_EM"].to_numpy()

In [ ]:
caloNu_tot_EM = np.nan_to_num(caloNu_tot_EM, nan=-1)
caloNu_tot_EM = np.where(caloNu_tot_EM > 1e7, 1e7, caloNu_tot_EM)
caloNu_tot_EM = caloNu_tot_EM / 1000  # convert to GeV

In [ ]:
front = [-2987.39, -1945.67]
middle = [-2965.18, -2246.18]
back = [-2246.18, -1946.18]

In [ ]:
# Get max energy for CaloNu for events in the front, middle, and back regions
front_caloNu = caloNu_tot_EM[(nu_vz > front[0]) & (nu_vz < front[1])]
middle_caloNu = caloNu_tot_EM[(nu_vz > middle[0]) & (nu_vz < middle[1])]
back_caloNu = caloNu_tot_EM[(nu_vz > back[0]) & (nu_vz < back[1])]

print(f"Front region: {front[0]} < z < {front[1]}, max CaloNu: {np.max(front_caloNu):.2f} GeV")
print(f"Middle region: {middle[0]} < z < {middle[1]}, max CaloNu: {np.max(middle_caloNu):.2f} GeV")
print(f"Back region: {back[0]} < z < {back[1]}, max CaloNu: {np.max(back_caloNu):.2f} GeV")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

bins_x = np.linspace(np.min(nu_vz), np.max(nu_vz), 100)
bins_y = np.logspace(np.log10(np.min(caloNu_tot_EM[caloNu_tot_EM > 0])), np.log10(np.max(caloNu_tot_EM)), 100)

ax.hist2d(nu_vz, caloNu_tot_EM, bins=[bins_x, bins_y], cmap="viridis", norm=LogNorm())
ax.axvline(x=front[0], color='r', linestyle='--', label="Front of CaloNu")
ax.axvline(x=front[1], color='r', linestyle='--')
ax.axvline(x=middle[0], color='g', linestyle='--', label="Middle of CaloNu")
ax.axvline(x=middle[1], color='g', linestyle='--')
ax.axvline(x=back[0], color='b', linestyle='--', label="Back of CaloNu")
ax.axvline(x=back[1], color='b', linestyle='--')
# ax.axhline(y=np.max(front_caloNu), color='r', linestyle=':', label="Max CaloNu in Front")
# ax.axhline(y=np.max(middle_caloNu), color='g', linestyle=':', label="Max CaloNu in Middle")
ax.axhline(y=np.max(back_caloNu), color='b', linestyle=':', label="Max CaloNu in Back")
ax.set_xlabel("Neutrino Decay Vertex Z (mm)")
ax.set_ylabel("CaloNu Total EM Energy (GeV)")
ax.set_yscale("log")
ax.legend()


In [ ]:
# Work out the efficiency of placing a cut on CaloNu energy to remove events in the back region

cut_value = np.max(back_caloNu)


purities = []
efficiencies = []
s_over_bs = []

for cut_value in np.linspace(0, np.max(back_caloNu), 100):
    # print(f"Cut value for CaloNu energy: {cut_value:.2f} GeV")

    front_caloNu = caloNu_tot_EM[(nu_vz > front[0]) & (nu_vz < front[1])]
    middle_caloNu = caloNu_tot_EM[(nu_vz > middle[0]) & (nu_vz < middle[1])]
    back_caloNu = caloNu_tot_EM[(nu_vz > back[0]) & (nu_vz < back[1])]

    number_of_true_front_events = len(front_caloNu)
    number_of_true_middle_events = len(middle_caloNu)
    number_of_true_back_events = len(back_caloNu)

    number_of_predicted_front_events = len(front_caloNu[front_caloNu < cut_value])
    number_of_predicted_middle_events = len(middle_caloNu[middle_caloNu < cut_value])
    number_of_predicted_back_events = len(back_caloNu[back_caloNu < cut_value])

    # print(f"Number of true front events: {number_of_true_front_events}")
    # print(f"Number of true middle events: {number_of_true_middle_events}")
    # print(f"Number of true back events: {number_of_true_back_events}")
    # print(f"Number of predicted front events: {number_of_predicted_front_events}")
    # print(f"Number of predicted middle events: {number_of_predicted_middle_events}")
    # print(f"Number of predicted back events: {number_of_predicted_back_events}")
    try:
        s_over_b = number_of_predicted_back_events / (number_of_predicted_front_events + number_of_predicted_middle_events)
        purity_back = number_of_predicted_back_events / (number_of_predicted_front_events + number_of_predicted_middle_events + number_of_predicted_back_events)
        efficiency_back = number_of_predicted_back_events / number_of_true_back_events
    except ZeroDivisionError:
        purity_back = 0
        efficiency_back = 0
        s_over_b = 0

    # print(f"Purity of back events: {purity_back:.2f}")
    # print(f"Efficiency of back events: {efficiency_back:.2f}")
    purities.append(purity_back)
    efficiencies.append(efficiency_back)
    s_over_bs.append(s_over_b)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(np.linspace(0, np.max(back_caloNu), 100), purities, label="Purity of Back Events")
ax.plot(np.linspace(0, np.max(back_caloNu), 100), efficiencies, label="Efficiency of Back Events")
ax.plot(np.linspace(0, np.max(back_caloNu), 100), s_over_bs, label="S/B of Back Events")

ax.axvline(x=6.7, color='r', linestyle='--', label=f"Cut Value: {cut_value:.2f} GeV")

ax.set_xlabel("CaloNu Energy Cut Value (GeV)")
ax.set_ylabel("Purity / Efficiency / S/B")
ax.set_title("Purity and Efficiency of Back Events vs CaloNu Energy Cut Value")
ax.legend()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
bins_x = np.linspace(np.min(nu_vz), np.max(nu_vz), 100)

ax.hist(nu_vz, bins=bins_x, color='blue', alpha=0.7, histtype='step', label="Neutrino Decay Vertex Z Distribution")
ax.axvline(x=front[0], color='r', linestyle='--', label="Front of CaloNu")
ax.axvline(x=front[1], color='r', linestyle='--')
ax.axvline(x=middle[0], color='g', linestyle='--', label="Middle of CaloNu")
ax.axvline(x=middle[1], color='g', linestyle='--')
ax.axvline(x=back[0], color='b', linestyle='--', label="Back of CaloNu")
ax.axvline(x=back[1], color='b', linestyle='--')
ax.set_xlabel("Neutrino Decay Vertex Z (mm)")
ax.set_ylabel("Number of Events")  

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
bins_x = np.linspace(np.min(caloNu_tot_EM), np.max(caloNu_tot_EM), 100)
ax.hist(caloNu_tot_EM, bins=bins_x, color='orange', alpha=0.7, histtype='step', label="CaloNu Total EM Energy Distribution")
ax.axvline(x=np.max(front_caloNu), color='r', linestyle=':', label="Max CaloNu in Front")
ax.axvline(x=np.max(middle_caloNu), color='g', linestyle=':', label="Max CaloNu in Middle")
ax.axvline(x=np.max(back_caloNu), color='b', linestyle=':', label="Max CaloNu in Back")
ax.set_xlabel("CaloNu Total EM Energy (GeV)")
ax.set_ylabel("Number of Events")
ax.set_yscale("log")
ax.legend()

In [ ]:
data_files = glob.glob("AnalysisZipFramework/submission/analysis_output/*.root")
calonu_data_files = []
for file in data_files:
    run_num = int(os.path.basename(file).split('.')[0])
    if runs_caloNu[0] <= run_num <= runs_caloNu[1]:
        calonu_data_files.append(f"{file}:nt")

In [ ]:
print(f"Found {len(calonu_data_files)} CaloNu data files in the range {runs_caloNu[0]} to {runs_caloNu[1]}.")

In [ ]:
df_data = uproot.concatenate(calonu_data_files, library="pd")

In [ ]:

mc_files_nominal = ["/home/ppd/bewilson/work/nuAnalysis/build/200139.1.root:nt",
         "/home/ppd/bewilson/work/nuAnalysis/build/200140.1.root:nt",
         "/home/ppd/bewilson/work/nuAnalysis/build/200146.1.root:nt",]

mc_files_FTF_BIC = ["/home/ppd/bewilson/work/nuAnalysis/build/200172.root:nt",
                    "/home/ppd/bewilson/work/nuAnalysis/build/200173.root:nt",
                    "/home/ppd/bewilson/work/nuAnalysis/build/200174.root:nt",]

mc_files_FTFP_INCLXX = ["/home/ppd/bewilson/work/nuAnalysis/build/200175.root:nt",
                        "/home/ppd/bewilson/work/nuAnalysis/build/200176.root:nt",
                        "/home/ppd/bewilson/work/nuAnalysis/build/200177.root:nt",]

mc_files_FTFP_BERT_HP_JEFF33 = ["/home/ppd/bewilson/work/nuAnalysis/build/200178.root:nt",
                               "/home/ppd/bewilson/work/nuAnalysis/build/200179.root:nt",
                               "/home/ppd/bewilson/work/nuAnalysis/build/200180.root:nt",]

mc_files_FTFP_BERT_HP_ENDFB8 = ["/home/ppd/bewilson/work/nuAnalysis/build/200181.root:nt",
                               "/home/ppd/bewilson/work/nuAnalysis/build/200182.root:nt",
                               "/home/ppd/bewilson/work/nuAnalysis/build/200183.root:nt",]


df_mc = uproot.concatenate(mc_files_nominal, library="pd")

In [ ]:
n_events_data = len(df_data)
n_events_mc = len(df_mc) * (70.03 / 10_000)

print(f"Number of events in data {n_events_data}")
print(f"Number of events in MC {n_events_mc:.2f}")
print(f"Ratio of data to MC = {n_events_data / n_events_mc:.2f}")
print(f"Number of events in MC (raw) = {len(df_mc)}")

In [ ]:
print(9084 + 7499 + 1116)
print(17699 * (70.03 / 10_000))

In [ ]:
mc1 = 29922 + 34949 + 3884
print(f"Number of events in MC (first sample) = {mc1}")

mc2  = 99802
print(f"Number of events in MC (second sample) = {mc2}")

In [ ]:
mc1 = 146711 + 628184 + 460000
mc2 = 1234895
print(f"Number of events in MC (first sample) = {mc1}")
print(f"Number of events in MC (second sample) = {mc2}")

In [ ]:
mc1 = 264889 + 351337 + 80407
mc2 = 933845
print(f"Number of events in MC (first sample) = {mc1}")
print(f"Number of events in MC (second sample) = {mc2}")

In [ ]:
status_cols = ["CaloNu0_status", "CaloNu1_status", "CaloNu2_status", "CaloNu3_status"]

df_data["Saturated_CaloNu_Module"] = (df_data[status_cols] == 4).any(axis=1).copy()
df_mc["Saturated_CaloNu_Module"] = (df_mc[status_cols] == 4).any(axis=1).copy()

In [ ]:
caloNu_tot_EM_data = df_data["CaloNu_total_E_EM"].to_numpy()
caloNu_tot_EM_data = np.nan_to_num(caloNu_tot_EM_data, nan=-1)
caloNu_tot_EM_data = np.where(caloNu_tot_EM_data > 1e7, 1e7, caloNu_tot_EM_data)
caloNu_tot_EM_data = caloNu_tot_EM_data / 1000  # convert to GeV

caloNu_tot_EM_mc = df_mc["CaloNu_total_E_EM"].to_numpy()
caloNu_tot_EM_mc = np.nan_to_num(caloNu_tot_EM_mc, nan=-1)
caloNu_tot_EM_mc = np.where(caloNu_tot_EM_mc > 1e7, 1e7, caloNu_tot_EM_mc)
caloNu_tot_EM_mc = caloNu_tot_EM_mc / 1000  # convert to GeV

In [ ]:
df_data["Average_CaloNu_localtime"] = df_data[["CaloNu0_localtime", "CaloNu1_localtime", "CaloNu2_localtime", "CaloNu3_localtime"]].mean(axis=1).copy()
df_mc["Average_CaloNu_localtime"] = df_mc[["CaloNu0_localtime", "CaloNu1_localtime", "CaloNu2_localtime", "CaloNu3_localtime"]].mean(axis=1).copy()

df_data["Min_CaloNu_localtime"] = df_data[["CaloNu0_localtime", "CaloNu1_localtime", "CaloNu2_localtime", "CaloNu3_localtime"]].min(axis=1).copy()
df_mc["Min_CaloNu_localtime"] = df_mc[["CaloNu0_localtime", "CaloNu1_localtime", "CaloNu2_localtime", "CaloNu3_localtime"]].min(axis=1).copy()

df_data["Max_CaloNu_localtime"] = df_data[["CaloNu0_localtime", "CaloNu1_localtime", "CaloNu2_localtime", "CaloNu3_localtime"]].max(axis=1).copy()
df_mc["Max_CaloNu_localtime"] = df_mc[["CaloNu0_localtime", "CaloNu1_localtime", "CaloNu2_localtime", "CaloNu3_localtime"]].max(axis=1).copy()

df_data["Average_CaloNu_bcidtime"] = df_data[["CaloNu0_bcidtime", "CaloNu1_bcidtime", "CaloNu2_bcidtime", "CaloNu3_bcidtime"]].mean(axis=1).copy()
df_mc["Average_CaloNu_bcidtime"] = df_mc[["CaloNu0_bcidtime", "CaloNu1_bcidtime", "CaloNu2_bcidtime", "CaloNu3_bcidtime"]].mean(axis=1).copy()

df_data["Average_CaloNu_triggertime"] = df_data[["CaloNu0_triggertime", "CaloNu1_triggertime", "CaloNu2_triggertime", "CaloNu3_triggertime"]].mean(axis=1).copy()
df_mc["Average_CaloNu_triggertime"] = df_mc[["CaloNu0_triggertime", "CaloNu1_triggertime", "CaloNu2_triggertime", "CaloNu3_triggertime"]].mean(axis=1).copy()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

bins = np.logspace(0.1, 3.5, 15)


h_mc = bh.Histogram(bh.axis.Variable(bins))
h_mc.fill(caloNu_tot_EM_mc, weight=70.03 / 10_000 ) # Normalise to lumi

h_data = bh.Histogram(bh.axis.Variable(bins))
h_data.fill(caloNu_tot_EM_data)  # convert to GeV

mplhep.histplot(h_data, ax=ax, label="CaloNu Data", color='orange', histtype='errorbar', linewidth=2, flow='show')
mplhep.histplot(h_mc, ax=ax, label="CaloNu MC", color='blue', histtype='step', linewidth=2, flow='show')

ax.set_xlabel("CaloNu Total EM Energy (GeV)")
ax.set_ylabel("Number of Events")
ax.set_yscale("log")
ax.set_xscale("log")
ax.legend()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
vz_mc = np.array([x[0] for x in df_mc["truth_dec_z"].to_numpy()])
ax.hist2d(vz_mc, df_mc["Average_CaloNu_localtime"].to_numpy(), bins=[100, 100], cmap="viridis", norm=LogNorm());
ax.set_xlabel("Neutrino Decay Vertex Z (mm)")
ax.set_ylabel("Average CaloNu Local Time (ns)")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
vz_mc = np.array([x[0] for x in df_mc["truth_dec_z"].to_numpy()])
ax.hist2d(vz_mc, df_mc["Min_CaloNu_localtime"].to_numpy(), bins=[100, 100], cmap="viridis", norm=LogNorm());
ax.set_xlabel("Neutrino Decay Vertex Z (mm)")
ax.set_ylabel("Minimum CaloNu Local Time (ns)")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
vz_mc = np.array([x[0] for x in df_mc["truth_dec_z"].to_numpy()])
ax.hist2d(vz_mc, df_mc["Max_CaloNu_localtime"].to_numpy(), bins=[100, 100], cmap="viridis", norm=LogNorm());
ax.set_xlabel("Neutrino Decay Vertex Z (mm)")
ax.set_ylabel("Maximum CaloNu Local Time (ns)")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 6))

df_mc_saturated = df_mc[df_mc["Saturated_CaloNu_Module"] == True]
df_mc_not_saturated = df_mc[df_mc["Saturated_CaloNu_Module"] == False]

vz_mc_saturated = np.array([x[0] for x in df_mc_saturated["truth_dec_z"].to_numpy()])
vz_mc_not_saturated = np.array([x[0] for x in df_mc_not_saturated["truth_dec_z"].to_numpy()])

ax.hist(vz_mc_saturated, bins=100, color='red', alpha=0.7, histtype='step', label="Saturated CaloNu Events");
ax.hist(vz_mc_not_saturated, bins=100, color='blue', alpha=0.7, histtype='step', label="Not Saturated CaloNu Events");

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(df_mc["CaloNu0_status"])
ax.set_yscale("log")
print(set(df_mc["CaloNu0_status"].to_numpy()))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(df_data["Preshower0_status"])
ax.set_yscale("log")
print(set(df_data["Preshower0_status"].to_numpy()))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(df_data["Preshower1_status"])
ax.set_yscale("log")
print(set(df_data["Preshower1_status"].to_numpy()))

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
ax.hist(caloNu_tot_EM_mc, bins=100, color='blue', alpha=0.7, histtype='step', label="CaloNu MC Total EM Energy Distribution")
ax.set_xlabel("CaloNu Total EM Energy (GeV)")
ax.set_ylabel("Number of Events")
ax.set_yscale("log")

In [ ]:
cut_val = 6.7

df_data_below = df_data[df_data["CaloNu_total_E_EM"] < cut_val * 1000]
df_data_above = df_data[df_data["CaloNu_total_E_EM"] >= cut_val * 1000]

df_mc_below = df_mc[df_mc["CaloNu_total_E_EM"] < cut_val * 1000]
df_mc_above = df_mc[df_mc["CaloNu_total_E_EM"] >= cut_val * 1000]

bins = np.linspace(-1, 30, 31)

h_data_reduced_charge0 = bh.Histogram(bh.axis.Variable(bins))
h_data_reduced_charge0.fill(df_data["VetoNu0_reduced_charge"].to_numpy())

h_data_reduced_charge0_below = bh.Histogram(bh.axis.Variable(bins))
h_data_reduced_charge0_below.fill(df_data_below["VetoNu0_reduced_charge"].to_numpy())

h_data_reduced_charge0_above = bh.Histogram(bh.axis.Variable(bins))
h_data_reduced_charge0_above.fill(df_data_above["VetoNu0_reduced_charge"].to_numpy())

h_mc_reduced_charge0 = bh.Histogram(bh.axis.Variable(bins))
h_mc_reduced_charge0.fill(df_mc["VetoNu0_reduced_charge"].to_numpy(), weight=70.03 / 10_000 ) # Normalise to lumi

h_mc_reduced_charge0_above = bh.Histogram(bh.axis.Variable(bins))
h_mc_reduced_charge0_above.fill(df_mc_above["VetoNu0_reduced_charge"].to_numpy(), weight=70.03 / 10_000 ) # Normalise to lumi

h_mc_reduced_charge0_below = bh.Histogram(bh.axis.Variable(bins))
h_mc_reduced_charge0_below.fill(df_mc_below["VetoNu0_reduced_charge"].to_numpy(), weight=70.03 / 10_000 ) # Normalise to lumi


print(f"Number of events in data below cut: {len(df_data_below)}, number of events in MC below cut: {len(df_mc_below)*70.03 / 10_000:.2f}")
print(f"Number of events in data above cut: {len(df_data_above)}, number of events in MC above cut: {len(df_mc_above)*70.03 / 10_000:.2f}")


In [ ]:
n_events_data = len(df_data)
n_events_mc = len(df_mc) * 70.03 / 10_000

print(f"Number of events in data {n_events_data}")
print(f"Number of events in MC {n_events_mc:.2f}")
print(f"Ratio of data to MC = {n_events_data / n_events_mc:.2f}")

In [ ]:
rate_data = (n_events_data / 70.03) / 0.68
rate_mc = (n_events_mc / 10_000) / 0.68

In [ ]:



fig, ax = plt.subplots(nrows=2, ncols=1, figsize=(10, 6))
mplhep.histplot(h_data_reduced_charge0, ax=ax[0], label="CaloNu Data", color='orange', histtype='errorbar', linewidth=2, flow='show')
mplhep.histplot(h_mc_reduced_charge0, ax=ax[0], label="CaloNu MC", color='blue', histtype='step', linewidth=2, flow='show')
ax[0].set_yscale("log")
ax[0].set_xlabel("VetoNu0 Reduced Charge")
ax[0].set_title("In CaloNu back region")
# Ratio plot
ratio = h_data_reduced_charge0 / h_mc_reduced_charge0

ratio_hist = h_data_reduced_charge0.view() / h_mc_reduced_charge0.view()
ratio_hist = bh.Histogram(bh.axis.Regular(bins.size - 1, bins[0], bins[-1]))
ratio_hist.view()[:] = ratio


mplhep.histplot(ratio_hist, ax=ax[1], histtype="errorbar", color="black")
ax[1].set_ylabel("Data / MC")

In [ ]:



fig, ax = plt.subplots(nrows=2, ncols=1, figsize=(10, 6))
mplhep.histplot(h_data_reduced_charge0_below, ax=ax[0], label="CaloNu Data Below Cut", color='orange', histtype='errorbar', linewidth=2, flow='show')
mplhep.histplot(h_mc_reduced_charge0_below, ax=ax[0], label="CaloNu MC Below Cut", color='blue', histtype='step', linewidth=2, flow='show')
ax[0].set_yscale("log")
ax[0].set_xlabel("VetoNu0 Reduced Charge")
ax[0].set_title("In CaloNu back region")
# Ratio plot
ratio = h_data_reduced_charge0_below / h_mc_reduced_charge0_below

ratio_hist = h_data_reduced_charge0_below.view() / h_mc_reduced_charge0_below.view()
ratio_hist = bh.Histogram(bh.axis.Regular(bins.size - 1, bins[0], bins[-1]))
ratio_hist.view()[:] = ratio


mplhep.histplot(ratio_hist, ax=ax[1], histtype="errorbar", color="black")
ax[1].set_ylabel("Data / MC")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), nrows=2)
mplhep.histplot(h_data_reduced_charge0_above, ax=ax[0], label="CaloNu Data Above Cut", color='red', histtype='errorbar', linewidth=2, flow='show')
mplhep.histplot(h_mc_reduced_charge0_above, ax=ax[0], label="CaloNu MC Above Cut", color='green', histtype='step', linewidth=2, flow='show')
ax[0].set_xlabel("VetoNu0 Reduced Charge")
ax[0].set_title("In CaloNu front/middle region")

ratio_hist = h_data_reduced_charge0_above.view() / h_mc_reduced_charge0_above.view()
ratio_hist = bh.Histogram(bh.axis.Regular(bins.size - 1, bins[0], bins[-1]))
ratio_hist.view()[:] = ratio

mplhep.histplot(ratio_hist, ax=ax[1], histtype="errorbar", color="black")
ax[1].set_ylabel("Data / MC")

